# Chapter 03-07 · Vectors, distance, and shapes

**Label:** Core  |  **Time:** ~55 minutes  |  **Difficulty:** moderate - the arithmetic is easy, and
one result in the middle is genuinely surprising

**Prerequisites:** 03-06 for slopes, and 01-03 for numpy arrays.

**Position in the learning path:** module 03, chapter 7 of 8. Before: **03-06**. After: **03-08**,
gradients - the last piece before models.

---

## Why this matters

One number per row was enough for six chapters. Real rows have many, and "many numbers at once" needs
a vocabulary. That vocabulary buys three things:

- **Distance**, which is what "similar" means to a computer - and therefore the entire basis of
  nearest-neighbours (06), clustering (08) and recommenders (11).
- **The dot product**, which turns out to be exactly what a linear model computes. Once you see this,
  the whole of module 05 is one line of arithmetic repeated.
- **Shapes**, which is what most of your error messages will be about.

There is also a failure lab in the middle that changes how you will treat every dataset. Below, an
obviously wrong flat comes out as the nearest match - not because of a bug, but because one column is
measured in thousands and another in single digits, and distance has no way to know that matters.

## What you will be able to do

- Treat a row of data as a point, and compute the distance between two rows
- Explain why distance is meaningless on unscaled data, and quantify how badly
- Say what a norm is, and when L1 and L2 disagree
- Recognise the dot product as a weighted sum, and read it as a prediction
- Say when to use cosine similarity instead of distance
- Read a shape error and know which array to fix

## Warm-up: retrieve, do not reread

1. What must accompany a slope for it to mean anything?
2. What does a coefficient of 0.70 on a logged outcome mean, as a percentage?
3. What does centring do to a slope, and to an intercept?

<br>

*Answers: (1) its units - rentals per degree, not "15". (2) 101.38%, not 70%. (3) nothing to the
slope; it makes the intercept the prediction at the average rather than at zero.*

## A row is a point

Five flats, three columns each. A **vector** is just a row: an ordered list of numbers. Three numbers
per flat means each flat is a point in three-dimensional space, where the axes are rent, rooms and
distance from the centre.

You are looking for somewhere to live. Your requirements are also a vector.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

flats = pd.DataFrame({
    "flat":  ["A", "B", "C", "D", "E"],
    "rent":  [1200, 1250, 2000, 1180, 1900],   # euros per month
    "rooms": [2, 4, 2, 4, 4],                  # number of rooms
    "km":    [1.0, 5.0, 1.0, 6.0, 1.5],        # kilometres from the centre
})

features = ["rent", "rooms", "km"]
X = flats[features].to_numpy(dtype=float)
wanted = np.array([1210.0, 4.0, 5.0])          # about 1210 euros, 4 rooms, 5 km out

print(flats.to_string(index=False))
print("\nwhat you want:", wanted)

### Distance, by hand

The distance between two points is the same formula you met at school, extended to as many columns as
you have: **square each difference, add them up, take the square root.**

For flat B against what you want: the rent differs by 40, the rooms by 0, the distance by 0. So the
distance is `sqrt(40^2 + 0 + 0)` = 40.

### Predict before running

Flat A has 2 rooms and sits 1 km out. You asked for 4 rooms and 5 km out - A is wrong on both. Flat B
has exactly 4 rooms and is exactly 5 km out, and costs 40 euros more than you hoped.

**Which one does the distance formula call your nearest match?** Write your answer down.

In [ ]:
differences = X - wanted
distances = np.sqrt((differences ** 2).sum(axis=1))

report = flats[["flat"]].copy()
report["rent gap"] = differences[:, 0]
report["rooms gap"] = differences[:, 1]
report["km gap"] = differences[:, 2]
report["distance"] = distances.round(3)
print(report.to_string(index=False))
print()
print("nearest match:", flats["flat"][distances.argmin()])

## Failure lab: the nearest flat is the wrong flat

**Flat A wins, at a distance of 10.954.** A has two rooms when you asked for four, and sits 1 km from
the centre when you asked for 5. Flat B, which matches both of your non-price requirements exactly,
comes third - behind a flat that is wrong about everything except its price.

There is no bug. The formula did exactly what it was asked. Look at what it was made of.

In [ ]:
squared = differences ** 2
share = squared / squared.sum(axis=1, keepdims=True)

contribution = pd.DataFrame(share, columns=features, index=flats["flat"])
print("share of each flat's squared distance contributed by each column:")
print(contribution.round(6).to_string())

### Diagnosis: distance is measured in whatever unit is largest

**Rent contributes between 83% and 99.997% of every distance.** For flats C, D and E the rooms and
kilometres columns together account for less than a thousandth of the answer.

The reason is arithmetic, not conceptual. A difference of 40 euros - trivial, less than a coffee a
week - contributes `40^2 = 1600` to the sum. A difference of 2 rooms - the difference between a flat
you can live in and one you cannot - contributes `4`. **The formula is comparing 1600 against 4**, and
euros win by four hundred to one because euros come in bigger numbers.

Distance has no notion of importance. It cannot: nothing in the data says a room matters more than a
euro. What it does instead is weight each column by **how large its numbers happen to be**, which is a
property of the measuring unit and nothing else. Had rent been recorded in thousands of euros, the
answer would change.

### The fix: put every column on the same footing

Standardise - subtract each column's mean and divide by its standard deviation, exactly as in 01-03.
Every column then has a spread of 1, and a "one standard deviation difference" means the same amount
of unusualness in each.

In [ ]:
means, sds = X.mean(axis=0), X.std(axis=0)
X_scaled = (X - means) / sds
wanted_scaled = (wanted - means) / sds

scaled_distances = np.sqrt(((X_scaled - wanted_scaled) ** 2).sum(axis=1))

comparison = flats[["flat"]].copy()
comparison["raw distance"] = distances.round(3)
comparison["raw rank"] = distances.argsort().argsort() + 1
comparison["scaled distance"] = scaled_distances.round(4)
comparison["scaled rank"] = scaled_distances.argsort().argsort() + 1
print("column means", means.round(2), " standard deviations", sds.round(3))
print()
print(comparison.to_string(index=False))
print()
print("nearest on raw data    :", flats["flat"][distances.argmin()])
print("nearest on scaled data :", flats["flat"][scaled_distances.argmin()])

**Scaled, flat B wins at 0.1097 and flat A drops to fourth.** That is the answer a person would give.

**The rule this establishes, and it is not optional:** any method that uses distances needs its
features on comparable scales. That includes k-nearest-neighbours, k-means, hierarchical clustering,
PCA, anomaly detection by distance, and any recommender built on similarity. Feeding raw columns to
these methods does not produce an error and does not produce a warning - it produces an answer
decided by your choice of units.

Two things to carry forward:

- **Scale using the training data only**, and apply the same shift and divisor to everything else.
  Computing the mean over your test set as well is leakage, and module 04 explains why that matters
  more than it sounds.
- **Standardising is a choice, not a neutral act.** It declares that one standard deviation of rent
  is as important as one standard deviation of rooms - which is a guess, and a much better one than
  "one euro equals one room", but still a guess. When you have real knowledge about importance,
  weighting the columns deliberately beats standardising.

**Tree-based methods are the exception.** Decision trees, random forests and gradient boosting split
one column at a time and never compare across columns, so scaling changes nothing for them at all.
That is one of the reasons they are so convenient in practice, and it is worth knowing which of your
tools care.

## Norms: more than one way to be far away

The formula above - square, add, root - is one choice among several. It is called the **L2 norm**, or
Euclidean distance, and it measures straight-line distance.

The **L1 norm** adds the absolute differences instead: no squaring, no root. It is also called
Manhattan distance, because it is how far you walk on a grid of streets.

You have met this fork before. In 03-01, the mean minimised squared error and the median minimised
absolute error. **These are the same two choices**, applied to distance rather than to a summary, and
the difference between them is the same: squaring punishes one large gap much more than several small
ones.

Here is a case where they disagree.

In [ ]:
target = np.array([0.0, 0.0, 0.0])
candidate_p = np.array([3.0, 0.0, 0.0])     # badly wrong on one feature
candidate_q = np.array([1.8, 1.8, 0.0])     # moderately wrong on two

for name, candidate in [("P - out by 3.0 on one feature ", candidate_p),
                        ("Q - out by 1.8 on two features", candidate_q)]:
    l1 = np.abs(candidate - target).sum()
    l2 = np.sqrt(((candidate - target) ** 2).sum())
    print("%s   L1 %.3f   L2 %.3f" % (name, l1, l2))

print()
print("L1 prefers P;  L2 prefers Q")
print("L2 for Q by hand: sqrt(1.8^2 + 1.8^2) = sqrt(%.2f) = %.4f" % (2 * 1.8 ** 2, np.sqrt(2 * 1.8 ** 2)))

### Which to use

**L2 prefers a candidate that is moderately wrong about several things. L1 prefers one that is
exactly right about most things and badly wrong about one.**

That is the whole distinction, and which you want depends on the problem:

- **L2** when errors compound or when being wildly wrong about anything is unacceptable. It is the
  default nearly everywhere, partly because it is smooth and differentiable - which matters for 03-08.
- **L1** when features are on their own terms and a single mismatch should not dominate. It is also
  more robust to a single bad measurement, for exactly the reason the median was.

They usually agree on the nearest neighbour and disagree at the margins, which is enough to change a
recommendation, a cluster assignment, or a shortlist. The habit is to know which one your library is
using - `sklearn`'s distance-based methods default to L2 and take a `metric` argument.

**A norm is also how you measure the size of a single vector**, by taking its distance from the
origin. `np.linalg.norm(v)` does that, and the same L1/L2 choice appears again in module 05 as Lasso
and Ridge - which are nothing more than a penalty on the L1 or L2 size of the coefficient vector, and
behave differently for the reason above.

## The dot product, which is what a model computes

Multiply two vectors element by element and add up the results. That is the dot product, written
`a @ b` in numpy, and it is the single most common operation in machine learning.

The reason it matters: **a weighted sum is a prediction**. Give each feature a weight, multiply, add,
and you have a linear model.

In [ ]:
weights = np.array([0.02, 30.0, -8.0])   # per euro, per room, per km

print("weights: %.2f per euro of rent, %.0f per room, %.0f per km from the centre"
      % (weights[0], weights[1], weights[2]))
print()
print("flat A by hand: 1200 x 0.02  +  2 x 30  +  1.0 x -8  =  %.1f"
      % (1200 * 0.02 + 2 * 30 + 1.0 * -8))
print("flat A by dot product:", X[0] @ weights)
print()
scores = X @ weights
print(pd.DataFrame({"flat": flats["flat"], "score": scores.round(2)}).to_string(index=False))

`X @ weights` computed all five predictions at once. That single line **is** a linear model's
prediction step - module 05 spends its time on how the weights are chosen, not on what happens
afterwards, because what happens afterwards is this.

Read the expression aloud and it stays intuitive: *"two hundredths of a point per euro, thirty points
per room, minus eight points per kilometre from the centre."* Each weight is a slope, in the sense of
03-06, with its own units - and that is precisely why comparing weights directly does not tell you
which feature matters most.

### The other thing a dot product measures: alignment

A dot product is large when two vectors point the same way and small when they do not. Divide by both
lengths and you get **cosine similarity**, which ranges from 1 for the same direction to -1 for
opposite directions, and ignores magnitude entirely.

That last property is the point. Consider three users and how many times each listened to two genres.

In [ ]:
def cosine(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))


you = np.array([2.0, 1.0])          # 2 folk, 1 techno
heavy_user = np.array([20.0, 10.0])  # the same taste, ten times the listening
different = np.array([1.0, 2.0])     # opposite taste, similar amount of listening

for name, other in [("heavy user, same ratio      ", heavy_user),
                    ("light user, opposite ratio  ", different)]:
    print("%s  euclidean %7.3f   cosine %.4f"
          % (name, np.linalg.norm(you - other), cosine(you, other)))

**Euclidean distance says the heavy user is far away (20.125) and the opposite-taste user is close
(1.414). Cosine says the heavy user is identical (1.0000) and the other is not (0.8000).**

For recommending music, cosine is obviously right: the heavy user has exactly your taste and simply
listens more, and their playlist is the one you want. Euclidean distance is measuring *how much they
listen*, which is not the question.

**The rule:** when magnitude reflects something you do not care about - activity level, document
length, image brightness - compare directions with cosine. When magnitude is part of what you are
comparing - rent, temperature, dosage - use distance. Module 11 builds a recommender on exactly this
distinction.

## Matrices and shapes

A matrix is a stack of vectors: `X` above is 5 rows by 3 columns, written `(5, 3)`. Almost every error
you will see in numpy or sklearn is about these two numbers not lining up.

**The one rule:** in `A @ B`, the columns of `A` must equal the rows of `B`, and what comes out has
the rows of `A` and the columns of `B`.

`(5, 3) @ (3,) -> (5,)`. Five rows in, one prediction each out.

In [ ]:
print("X       ", X.shape)
print("weights ", weights.shape)
print("X @ w   ", (X @ weights).shape, "  <- one number per flat")
print()
column_weights = weights.reshape(-1, 1)
print("weights as a column", column_weights.shape)
print("X @ column_weights ", (X @ column_weights).shape, "  <- a COLUMN of results, not a flat list")
print()
print("X.T     ", X.T.shape)
print("X.T @ X ", (X.T @ X).shape, "  <- one entry per pair of features")

In [ ]:
try:
    X @ np.array([1.0, 2.0])          # only two weights for three columns
except ValueError as error:
    print("ValueError:", error)

### Reading that message

It is verbose, and the useful part is at the end: **"size 2 is different from 3"**. `X` has 3 columns
and the vector has 2 entries, so the two cannot meet.

A method for shape errors that works nearly every time:

1. **Print the shapes** of both operands. The mismatch is almost always obvious once you can see them.
2. **Decide which one is wrong.** Usually the data is right and the thing you built by hand is wrong.
3. **Check for the `(n,)` versus `(n, 1)` confusion**, which is the most common cause. A
   one-dimensional array of five numbers is `(5,)`; a column of five numbers is `(5, 1)`. They behave
   differently under broadcasting, and 01-03's failure lab was exactly this.

**`X.T @ X` is worth recognising** - `(3, 5) @ (5, 3)` gives `(3, 3)`, one entry per pair of features.
It appears in the closed-form solution for linear regression in module 05, and its size depends only
on the number of features, which is why that solution is fast with many rows and slow with many
columns.

## Common misconceptions

**"Distance is objective."**
Distance is measured in whatever units your columns happen to use. On the flats data, rent supplied
between 83% and 99.997% of every distance, and the "nearest" flat had the wrong number of rooms.

**"Standardising is a preprocessing detail."**
For any distance-based method it decides the answer. It is as much a modelling choice as picking the
algorithm.

**"Scaling always helps."**
It does nothing for trees, forests and boosting, which never compare columns to each other. Knowing
which of your methods are geometric and which are not saves a lot of pointless pipeline work.

**"L1 and L2 are interchangeable."**
They agree often and disagree exactly where it matters - one is prepared to accept a single large
error to keep the others small, and the other is not.

**"The dot product is an abstract operation."**
It is a weighted sum, and a weighted sum is a prediction. `X @ w` is a linear model's entire
prediction step.

**"Cosine similarity is a better distance."**
It is a different question. It ignores magnitude, which is right for taste and wrong for rent.

**"Shape errors mean my data is broken."**
They usually mean an array you constructed is `(n,)` where something expected `(n, 1)`, or the other
way round. Print both shapes before reading the message.

## Exercises

Solutions: `solutions/03_math_foundations/03-07_vectors_matrices_solutions.ipynb`.

### Quick understanding

**E1.** In one sentence, why did the raw distance pick a flat with the wrong number of rooms?

**E2.** Name two methods for which scaling changes the answer and two for which it changes nothing.

**E3.** What does cosine similarity ignore, and give one situation where ignoring it is right and one
where it is wrong.

### Hand calculation

**E4.** Compute by hand the Euclidean and Manhattan distance between `[3, 4, 0]` and `[0, 0, 12]`.
Which is larger, and will that always be true?

**E5.** A model has weights `[0.5, -2.0, 10.0]` for features `[area_m2, age_years, has_garden]`.
Compute the prediction for a 80 m2, 12-year-old flat with a garden. Then say what would happen to the
weights if area were recorded in square centimetres instead.

**E6.** Two users have listening vectors `[4, 2]` and `[1, 8]`. Compute their cosine similarity by
hand. Then compute it for `[4, 2]` and `[8, 4]` and explain the result in one sentence.

### Coding

**E7.** Write `nearest(X, query, scale=True)` returning the index of the nearest row, standardising
first when asked. Run it on the flats data both ways and confirm it reproduces the chapter's flip.

**E8.** Take the flats data and record rent in **thousands** of euros instead of euros. Recompute the
raw distances. Which flat is nearest now? Explain what this proves about unscaled distance.

**E9.** Write `distance_matrix(X)` returning all pairwise Euclidean distances using broadcasting and
no Python loops. Check it against a loop version on the flats data, and state the shape and memory
cost for 10,000 rows.

### Interpretation

**E10.** A colleague builds a k-nearest-neighbours model on customer data with columns `age` (18-90),
`income` (15,000-200,000) and `visits_per_month` (0-30), without scaling. Predict which column decides
every neighbour, and estimate roughly what share of the squared distance it will contribute.

**E11.** A recommender uses Euclidean distance on raw play counts and its recommendations are
dominated by a handful of very active users appearing as nobody's neighbour. Explain the mechanism and
name the fix.

### Debugging

**E12.** An analyst writes `predictions = X @ weights` and gets `(5, 5)` instead of `(5,)`. Given
`X.shape == (5, 3)`, what shape must `weights` have been, and what did they most likely do wrong?

### Exam and interview reasoning

**E13.** "Why do you standardise features?" Answer in five sentences: what problem it solves, a case
where it is essential, a case where it is unnecessary, what it silently assumes, and how it must be
fitted when there is a train/test split.

### Transfer to a different situation

**E14.** You are matching patients to similar past cases using age (years), blood pressure (mmHg),
BMI, and number of prior admissions. Describe how you would set up the distance, including one column
you would deliberately weight more heavily and why standardising alone would not achieve that.

### Explain it to someone non-technical

**E15.** Explain, in under 90 words, why a computer asked to find "the most similar flat" returned one
with the wrong number of rooms - and what you changed.

### Optional challenge

**E16.** Generate points uniformly in a `d`-dimensional cube for `d` from 2 to 200, and for each `d`
compute the ratio of the distance to the *furthest* point to the distance to the *nearest*, averaged
over many query points. Plot it against `d`. What happens, and what does it imply for
nearest-neighbour methods on wide data?

In [ ]:
# Your workspace. In memory: flats, X, wanted, features, distances,
# X_scaled, wanted_scaled, scaled_distances, weights, cosine.

## Mastery check

- [ ] Compute a distance between two rows by hand, and say what it is measured in
- [ ] Explain why unscaled distance is decided by the largest-numbered column, with a percentage
- [ ] Say which methods need scaling and which are indifferent to it
- [ ] State the difference between L1 and L2 in terms of one large error versus several small ones
- [ ] Recognise `X @ w` as a linear model's prediction, and read the weights with their units
- [ ] Choose between cosine and distance for a stated problem
- [ ] Read a shape error and say which array to change

## What should now feel instinctive

- Asking about the units of every column before computing any distance
- Standardising before anything geometric, and fitting the scaler on training data only
- Seeing `X @ w` and reading it as "a prediction per row"
- Reaching for cosine the moment magnitude is not part of the question
- Printing shapes before reading a shape error

## Flashcards

| Front | Back |
|---|---|
| Euclidean distance | Square the differences, add, take the root. The L2 norm |
| Manhattan distance | Add the absolute differences. The L1 norm |
| L1 versus L2 | L2 prefers several small errors; L1 prefers one large one and the rest exact |
| Distance on unscaled data | Decided by the column with the largest numbers - 83% to 99.997% here |
| Methods needing scaling | kNN, k-means, PCA, SVM, distance-based anomaly detection |
| Methods indifferent to scaling | Trees, random forests, gradient boosting |
| Dot product | Element-by-element multiply and add. A weighted sum, and therefore a prediction |
| Cosine similarity | Dot product over both lengths - direction only, magnitude ignored |
| Shape rule for `A @ B` | Columns of A must equal rows of B; result is rows of A by columns of B |
| `(n,)` versus `(n, 1)` | A flat list versus a column. The commonest cause of shape errors |

## Next

**03-08 · Loss, gradients, and how a model is fitted.** You can now write a prediction as `X @ w` and
measure how wrong it is. The last chapter of the module is about how `w` gets chosen - which turns out
to be an idea you have already met twice, in 03-01's search for the number that minimised total error.